## Инициализайия и нормализация

В этом задании вам предстоит реализовать два вида нормализации: по батчам (BatchNorm1d) и по признакам (LayerNorm1d).

In [13]:
from typing import Callable, NamedTuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

### 1. Реализация BatchNorm1d и LayerNorm1d.

#### 1.1. (2 балла) Реализуйте BatchNorm1d

Подсказка: чтобы хранить текущие значения среднего и дисперсии, вам потребуется метод `torch.nn.Module.register_buffer`, ознакомьтесь с документацией к нему. Подумайте, какие проблемы возникнут, если вы будете просто сохранять ваши значения в тензор

In [14]:
class BatchNorm1d(nn.Module):
    def __init__(
        self, num_features: int, momentum: float = 0.9, eps: float = 1e-5
    ) -> None:
        super().__init__()
        self.scale = nn.Parameter(torch.ones(num_features))
        self.shift = nn.Parameter(torch.zeros(num_features))
        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))
        self.momentum = momentum
        self.eps = eps

    def forward(self, x: Tensor) -> Tensor:
        new_var = torch.var(x, unbiased=not self.training, dim=0)
        new_mean = x.mean(dim=0)
        with torch.no_grad():
            self.running_var = self.running_var * self.momentum + new_var * (1 - self.momentum)
            self.running_mean = self.running_mean * self.momentum + new_mean * (1 - self.momentum)
        return (x - self.running_mean) / (self.running_var + self.eps).sqrt() * self.scale + self.shift

#### 1.2. (1 балл) Реализуйте LayerNorm1d

Отличия LayerNorm от BatchNorm - в том, что расчёт средних и дисперсий в BatchNorm происходит вдоль размерности батча (см. рисунок слева), а в LayerNorm - вдоль размерности признаков (см. рисунок справа).

<img src="../attachments/norm.png" width="800">

In [15]:
class LayerNorm1d(nn.Module):
    def __init__(self, num_features: int, eps: float = 1e-5) -> None:
        super(LayerNorm1d, self).__init__()
        self.scale = nn.Parameter(torch.ones(num_features))
        self.shift = nn.Parameter(torch.zeros(num_features))
        self.eps = eps

    def forward(self, x: Tensor) -> Tensor:
        var = torch.var(x, dim=1, unbiased=False, keepdim=True)
        mean = x.mean(dim=1, keepdim=True)
        return (x - mean) / (var + self.eps).sqrt() * self.scale + self.shift

tens = torch.randn((100, 100))
theirs = nn.LayerNorm(100)(tens)
ours = LayerNorm1d(100)(tens)
ours.training = True
theirs.training = True
print(ours)
print(theirs)
print(torch.allclose(ours, theirs))

tensor([[ 0.5127, -0.0723,  1.0307,  ...,  0.1775,  0.2853,  0.4924],
        [ 0.1739, -0.0611,  0.5904,  ...,  0.2638,  0.3631,  0.7268],
        [-0.0939,  0.3827, -0.9113,  ...,  1.5629,  0.7634, -1.6151],
        ...,
        [ 0.7389, -1.1166,  1.8419,  ...,  0.3006,  0.5116,  1.3697],
        [-1.5470, -0.4905, -1.8931,  ..., -0.5922, -1.7993,  1.1126],
        [-1.2065, -0.4116, -0.8703,  ..., -0.2056,  1.0107,  0.8279]],
       grad_fn=<AddBackward0>)
tensor([[ 0.5127, -0.0723,  1.0307,  ...,  0.1775,  0.2853,  0.4924],
        [ 0.1739, -0.0611,  0.5904,  ...,  0.2638,  0.3631,  0.7268],
        [-0.0939,  0.3827, -0.9113,  ...,  1.5629,  0.7634, -1.6151],
        ...,
        [ 0.7389, -1.1166,  1.8419,  ...,  0.3006,  0.5116,  1.3697],
        [-1.5470, -0.4905, -1.8931,  ..., -0.5922, -1.7993,  1.1126],
        [-1.2065, -0.4116, -0.8703,  ..., -0.2056,  1.0107,  0.8279]],
       grad_fn=<NativeLayerNormBackward0>)
False


### 2. Эксперименты

В этом задании ваша задача - проверить, какие из приёмов хорошо справляются с нездоровыми активациями в промежуточных слоях. Вам будет дана базовая модель, у которой есть проблемы с инициализацией параметров, попробуйте несколько приёмов для устранения проблем обучения:
1. Хорошая инициализация параметров
2. Ненасыщаемая функция активации (например, `F.leaky_relu`)
3. Нормализация по батчам или по признакам (можно использовать встроенные `nn.BatchNorm1d` и `nn.LayerNorm`)
4. Более продвинутый оптимизатор (`torch.optim.RMSprop`)

#### 2.0. Подготовка: датасет, функции для обучения

Проверять наши гипотезы будем на датасете MNIST, для отладки добавим в функции для обучения возможность использовать только несколько батчей данных

In [16]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

train_dataset = datasets.MNIST(
    "data",
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)
test_dataset = datasets.MNIST(
    "data",
    train=False,
    download=True,
    transform=transforms.ToTensor(),
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [17]:
def training_step(
    batch: tuple[torch.Tensor, torch.Tensor],
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
) -> torch.Tensor:
    # прогоняем батч через модель
    x, y = batch
    logits = model(x)
    # оцениваем значение ошибки
    loss = F.cross_entropy(logits, y)
    # обновляем параметры
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    # возвращаем значение функции ошибки для логирования
    return loss


def train_epoch(
    dataloader: DataLoader,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    max_batches: int = 100,
) -> Tensor:
    loss_values: list[float] = []
    for i, batch in enumerate(dataloader):
        loss = training_step(batch, model, optimizer)
        loss_values.append(loss.item())
        if i == max_batches:
            break
    return torch.tensor(loss_values).mean()


@torch.no_grad()
def test_epoch(
    dataloader: DataLoader, model: nn.Module, max_batches: int = 100
) -> Tensor:
    loss_values: list[float] = []
    for i, batch in enumerate(dataloader):
        x, y = batch
        logits = model(x)
        # оцениваем значение ошибки
        loss = F.cross_entropy(logits, y)
        loss_values.append(loss.item())
        if i == max_batches:
            break
    return torch.tensor(loss_values).mean()

#### 2.1. Определение класса модели (2 балла)

Для удобства проведения экспериментов мы немного усложним создание модели, чтобы можно было задать разные способы инициализации параметров и нормализации промежуточных активаций, не меняя определение класса.

Добавьте в метод `__init__`:
- аргумент, который позволит использовать разные функции активации для промежуточных слоёв
- аргумент, который позволит задавать разные способы нормализации: `None` (без нормализации), `nn.BatchNorm` и `nn.LayerNorm`

In [18]:
def init_std_normal(model: nn.Module) -> None:
    """Функция для инициализации параметров модели стандартным нормальным распределением."""
    for param in model.parameters():
        torch.nn.init.normal_(param.data, mean=0, std=1)

def init_kaiming_normal(model: nn.Module) -> None:
    """Функция для инициализации параметров модели методом Кайминга с нормальным распределением."""
    for param in model.parameters():
        if isinstance(param, nn.Linear):
            nn.init.kaiming_normal_(param.data)


from typing import Type


class MLP(nn.Module):
    """Базовая модель для экспериментов

    Args:
        input_dim (int): размерность входных признаков
        hidden_dim (int): размерност скрытого слоя
        output_dim (int): кол-во классов
        act_fn (Callable[[Tensor], Tensor], optional): Функция активации. Defaults to F.tanh.
        init_fn (Callable[[nn.Module], None], optional): Функция для инициализации. Defaults to init_std_normal.
        norm (Type[nn.BatchNorm1d  |  nn.LayerNorm] | None, optional): Способ нормализации промежуточных активаций.
            Defaults to None.
    """
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        act_fn: Callable[[Tensor], Tensor] = F.tanh,
        init_fn: Callable[[nn.Module], None] = init_std_normal,
        norm: Type[nn.BatchNorm1d | nn.LayerNorm] | None = None,
    ) -> None:
        super().__init__()
        # теперь линейные слои будем задавать
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.act_fn = act_fn
        self.norm = None if norm is None else norm(hidden_dim)

        # reinitialize parameters
        init_fn(self)

    def forward(self, x: Tensor) -> Tensor:
        h = self.fc1.forward(x.flatten(1))
        # here you can do normalization
        if self.norm:
            h = self.norm(h)
        return self.fc2.forward(self.act_fn(h))

#### 2.2. Эксперименты (7 баллов)

Проведите по 3 эксперимента с каждой из модификаций с разными значениями `seed`, соберите статистику значений тестовой ошибки после 10 эпох обучения, сделайте выводы о том, что работает лучше

Проверяем:
1. Метод инициализации весов модели: $\mathcal{N}(0, 1)$ / Kaiming normal
2. Функция активации: tanh /  (или любая другая без насыщения)
3. Слой нормализации: None / BatchNorm / LayerNorm
4. Выбранный оптимизатор: SGD / RMSprop / Adam

Итого у нас 2 + 2 + 3 + 3 = 10 экспериментов, каждый нужно повторить 3 раза, посчитать среднее и вывести результаты в pandas.DataFrame.
Можно дополнительно потестировать разные сочетания опций, например инициализация + нормализация


Чтобы автоматизировать проведение экспериментов, можно использовать функцию, которая будет принимать все необходимые настройки эксперимента, запускать его и сохранять нужные метрики:

In [19]:
def run_experiment(
    model_gen: Callable[[], nn.Module],
    optim_gen: Callable[[nn.Module], torch.optim.Optimizer],
    seed: int,
    n_epochs: int = 10,
    max_batches: int | None = None,
    verbose: bool = False,
) -> float:
    """Функция для запуска экспериментов.

    Args:
        model_gen (Callable[[], nn.Module]): Функция для создания модели
        optim_gen (Callable[[nn.Module], torch.optim.Optimizer]): Функция для создания оптимизатора для модели
        seed (int): random seed
        n_epochs (int, optional): Число эпох обучения. Defaults to 10.
        max_batches (int | None, optional): Если указано, только `max_batches` минибатчей
            будет использоваться при обучении и тестировании. Defaults to None.
        verbose (bool, optional): Выводить ли информацию для отладки. Defaults to False.

    Returns:
        float: Значение ошибки на тестовой выборке в конце обучения
    """
    torch.manual_seed(seed)
    # создадим модель и выведем значение ошибки после инициализации
    model = model_gen()
    optim = optim_gen(model)
    epoch_losses: list[float] = []
    for i in range(n_epochs):
        train_loss = train_epoch(train_loader, model, optim, max_batches=max_batches)
        test_loss = test_epoch(test_loader, model, max_batches=max_batches)
        if verbose:
            print(f"Epoch {i} train loss = {train_loss:.4f}")
            print(f"Epoch {i} test loss = {test_loss:.4f}")

        epoch_losses.append(test_loss.item())

    last_epoch_loss = epoch_losses[-1]
    return last_epoch_loss

Пример использования:

In [20]:
losses = run_experiment(
    model_gen=lambda: MLP(784, 128, 10, init_fn=init_std_normal, norm=None),
    optim_gen=lambda x: torch.optim.SGD(x.parameters(), lr=0.01),
    seed=42,
    n_epochs=10,
    max_batches=100,
    verbose=True,
)

Epoch 0 train loss = 12.6168
Epoch 0 test loss = 9.9327
Epoch 1 train loss = 9.0954
Epoch 1 test loss = 7.5498
Epoch 2 train loss = 6.9607
Epoch 2 test loss = 6.2342
Epoch 3 train loss = 5.8992
Epoch 3 test loss = 5.3655
Epoch 4 train loss = 4.9951
Epoch 4 test loss = 4.7433
Epoch 5 train loss = 4.4778
Epoch 5 test loss = 4.3001
Epoch 6 train loss = 3.9693
Epoch 6 test loss = 3.9605
Epoch 7 train loss = 3.7261
Epoch 7 test loss = 3.6844
Epoch 8 train loss = 3.4223
Epoch 8 test loss = 3.4538
Epoch 9 train loss = 2.9975
Epoch 9 test loss = 3.2638


Для удобства задания настроек эксперимента можно определять их с помощью класса `Experiment`, в котором можно также реализовать логику для строкового представления:

In [21]:
input_dim = 784
hidden_dim = 128
output_dim = len(train_dataset.classes)


class Experiment(NamedTuple):
    init_fn: Callable[[nn.Module], None]
    act_fn: Callable[[Tensor], Tensor]
    norm: Type[nn.BatchNorm1d | nn.LayerNorm] | None
    optim_cls: Type[torch.optim.Optimizer]

    @property
    def model_gen(self) -> Callable[[], nn.Module]:
        return lambda: MLP(
            input_dim, hidden_dim, output_dim, init_fn=self.init_fn, norm=self.norm
        )

    @property
    def optim_gen(self) -> Callable[[nn.Module], torch.optim.Optimizer]:
        return lambda x: self.optim_cls(x.parameters(), lr=0.01)

    def __repr__(self) -> str:
        return (f"Experiment(init_fn={self.init_fn.__name__}, "
                f"act_fn={self.act_fn.__name__}, "
                f"norm={'None' if self.norm is None else self.norm.__name__}, "
                f"optim_cls={self.optim_cls.__name__})")



Описываем все эксперименты:

In [22]:
options = [
    Experiment(
        init_fn=init_kaiming_normal,
        act_fn=F.tanh,
        norm=None,
        optim_cls=torch.optim.RMSprop,
    ),
    Experiment(
        init_fn=init_std_normal,
        act_fn=F.silu,
        norm=nn.LayerNorm,
        optim_cls=torch.optim.SGD,
    ),
    Experiment(
        init_fn=init_std_normal,
        act_fn=F.relu,
        norm=nn.BatchNorm1d,
        optim_cls=torch.optim.RMSprop,
    ),
    Experiment(
        init_fn=init_std_normal,
        act_fn=F.silu,
        norm=nn.BatchNorm1d,
        optim_cls=torch.optim.SGD,
    ),
    Experiment(
        init_fn=init_kaiming_normal,
        act_fn=F.silu,
        norm=nn.LayerNorm,
        optim_cls=torch.optim.SGD,
    ),
    Experiment(
        init_fn=init_kaiming_normal,
        act_fn=F.tanh,
        norm=nn.BatchNorm1d,
        optim_cls=torch.optim.Adam,
    ),
    Experiment(
        init_fn=init_kaiming_normal,
        act_fn=F.silu,
        norm=None,
        optim_cls=torch.optim.Adam,
    ),
    Experiment(
        init_fn=init_std_normal,
        act_fn=F.relu,
        norm=None,
        optim_cls=torch.optim.Adam,
    ),
    Experiment(
        init_fn=init_kaiming_normal,
        act_fn=F.relu,
        norm=nn.LayerNorm,
        optim_cls=torch.optim.RMSprop,
    ),
    Experiment(
        init_fn=init_std_normal,
        act_fn=F.tanh,
        norm=nn.LayerNorm,
        optim_cls=torch.optim.SGD,
    ),
]

options

[Experiment(init_fn=init_kaiming_normal, act_fn=tanh, norm=None, optim_cls=RMSprop),
 Experiment(init_fn=init_std_normal, act_fn=silu, norm=LayerNorm, optim_cls=SGD),
 Experiment(init_fn=init_std_normal, act_fn=relu, norm=BatchNorm1d, optim_cls=RMSprop),
 Experiment(init_fn=init_std_normal, act_fn=silu, norm=BatchNorm1d, optim_cls=SGD),
 Experiment(init_fn=init_kaiming_normal, act_fn=silu, norm=LayerNorm, optim_cls=SGD),
 Experiment(init_fn=init_kaiming_normal, act_fn=tanh, norm=BatchNorm1d, optim_cls=Adam),
 Experiment(init_fn=init_kaiming_normal, act_fn=silu, norm=None, optim_cls=Adam),
 Experiment(init_fn=init_std_normal, act_fn=relu, norm=None, optim_cls=Adam),
 Experiment(init_fn=init_kaiming_normal, act_fn=relu, norm=LayerNorm, optim_cls=RMSprop),
 Experiment(init_fn=init_std_normal, act_fn=tanh, norm=LayerNorm, optim_cls=SGD)]

Запускаем расчёты:

In [11]:
seeds = [42, 43, 44]  # здесь вам нужно 3 разных значения
results = []

for option in options:
    print(option)
    for seed in seeds:
        loss = run_experiment(
            model_gen=option.model_gen,
            optim_gen=option.optim_gen,
            seed=seed,
            n_epochs=10,
            max_batches=None,
            verbose=True,
        )
        results.append([str(option), seed, loss])

Experiment(init_fn=init_kaiming_normal, act_fn=tanh, norm=None, optim_cls=RMSprop)
Epoch 0 train loss = 0.3353
Epoch 0 test loss = 0.1924
Epoch 1 train loss = 0.2033
Epoch 1 test loss = 0.1719
Epoch 2 train loss = 0.1808
Epoch 2 test loss = 0.1656
Epoch 3 train loss = 0.1633
Epoch 3 test loss = 0.1589
Epoch 4 train loss = 0.1510
Epoch 4 test loss = 0.1724
Epoch 5 train loss = 0.1472
Epoch 5 test loss = 0.1550
Epoch 6 train loss = 0.1343
Epoch 6 test loss = 0.1620
Epoch 7 train loss = 0.1352
Epoch 7 test loss = 0.1847
Epoch 8 train loss = 0.1290
Epoch 8 test loss = 0.1494
Epoch 9 train loss = 0.1284
Epoch 9 test loss = 0.1700
Epoch 0 train loss = 0.3119
Epoch 0 test loss = 0.1854
Epoch 1 train loss = 0.1982
Epoch 1 test loss = 0.2275
Epoch 2 train loss = 0.1817
Epoch 2 test loss = 0.1787
Epoch 3 train loss = 0.1653
Epoch 3 test loss = 0.1735
Epoch 4 train loss = 0.1563
Epoch 4 test loss = 0.2206
Epoch 5 train loss = 0.1493
Epoch 5 test loss = 0.1613
Epoch 6 train loss = 0.1378
Epoch 6 t

Выводим результаты:

In [23]:
import pandas as pd

# print(set(map(str, results)))
pd.DataFrame(results)

,0,1,2
0,"Experiment(init_fn=init_kaiming_normal, act_fn...",42,0.169976
1,"Experiment(init_fn=init_kaiming_normal, act_fn...",43,0.175457
2,"Experiment(init_fn=init_kaiming_normal, act_fn...",44,0.162758
3,"Experiment(init_fn=init_std_normal, act_fn=sil...",42,0.455568
4,"Experiment(init_fn=init_std_normal, act_fn=sil...",43,0.465426
5,"Experiment(init_fn=init_std_normal, act_fn=sil...",44,0.460751
6,"Experiment(init_fn=init_std_normal, act_fn=rel...",42,0.161209
7,"Experiment(init_fn=init_std_normal, act_fn=rel...",43,0.158187
8,"Experiment(init_fn=init_std_normal, act_fn=rel...",44,0.151873
9,"Experiment(init_fn=init_std_normal, act_fn=sil...",42,0.464949


ВЫВОДЫ:

Как всегд отложил до конца... Оформить выводы используя красивые таблички уже не успею...

Выводы такие, что в верхней половине в основном kaiming normal, и функция активации relu